# Distancia entre secuencias y árboles filogenéticos 

Un árbol filogenético es la representación de una hipótesis de **relaciones ancestrales**, basada en **similitud**. En esta representación, las especies o secuencias de interés estan en las puntas (*tips*) de las ramas. El nodo donde se unen las ramas representa el ancestro común. El tiempo por lo tanto va desde el ancestro común hacia las puntas, en la siguiente figura el tiempo sería el eje horizontal x  

<img src="Anatomy_tree.png"/>


Para reconstruir ese árbol, el procedimiento suele dividirse en dos grandes etapas:

1. **Cálculo de la matriz de distancias**
Partiendo de un alineamiento múltiple de secuencias (ver https://lauraalazar.github.io/Online_BioComp/alineamientos.html), podemos cuantificar cuán divergentes están entre sí usando diferentes modelos evolutivos. Estas distancias estiman la probabilidad acumulada de sustituciones entre pares de secuencias.

2. Reconstrucción del árbol a partir de la matriz de distancias
Dos métodos clásicos son UPGMA y Neighbour Joining


### UPGMA: Unweighted Pair Group Method with Arithmetic mean

El método UPGMA es un ejemplo de cluster jerárquico. La distancia entre dos clusters se calcula con la media de las distancias entre cada punto del primer cluster al segundo cluster

En el siguiente link puede encontrar un ejemplo paso a paso para construir el siguiente agrupamiento: http://www.slimsuite.unsw.edu.au/teaching/upgma/

<img src="upgma15.png"/>

### NJ: Neighbour Joining

Mientras que en un árbol realizado con UPGMA todos los puntos están alineados y las ramas tienen la misma distancia (porque las distancias se calculan por la media), en NJ se calcula la “distancia acumulada” para cada secuencia i (A-E),  $U_i = Σ d_{ij}$. Usando esta distancia acumulada Ui, se calcula una matriz de distancia $e_{ij}$ (para cada par de de secuencias i ay j), de acuerdo a la fórmula 

$e_{ij}$ = $d_{ij}$– $\frac{U_i + U_j}{N – 2}$

<img src="NJfig.png"/>

Para una explicación manual: https://www.tenderisthebyte.com/blog/2022/08/31/neighbor-joining-trees/

## Ejercicio con las secuencias de Citocromo B

A continuación vamos a importar las secuencias de DNA y proteína de Citocromo B que habíamos estudiado antes. A partir de este archivo vamos a:
- 1. Alinear
- 2. Calcular las distancias por pares 
- 3. Aplicar los algoritmos de agrupamiento
- 4. Graficar el árbol 

In [ ]:
#-----------------------------------------------------------
# 1. Instalar y cargar librerías
#-----------------------------------------------------------
if (!requireNamespace("ape", quietly = TRUE)) install.packages("ape")
if (!requireNamespace("phangorn", quietly = TRUE)) install.packages("phangorn")
if (!requireNamespace("msa", quietly = TRUE)) install.packages("msa")

library(ape)
library(phangorn)
library(msa)   # Para alinear las secuencias (multiple sequence aln)

#-----------------------------------------------------------
# 2. Leer las secuencias originales (sin alinear) desde GitHub
#-----------------------------------------------------------

# ADN
cytb_dna_raw <- readDNAStringSet(
  "https://raw.githubusercontent.com/lauraalazar/BiologiaComputacional/main/CytBDNA.txt",
  format = "fasta"
)

# Proteína
cytb_prot_raw <- readAAStringSet(
  "https://raw.githubusercontent.com/lauraalazar/BiologiaComputacional/main/CytBProt.txt",
  format = "fasta"
)

#-----------------------------------------------------------
# 3. Alinear las secuencias con MSA (ClustalW)
#-----------------------------------------------------------
alignment_dna <- msa(cytb_dna_raw, method = "ClustalW")
alignment_prot <- msa(cytb_prot_raw, method = "ClustalW")

# Convertir los alineamientos a formato DNAbin y AAbin
alignment_dna_bin <- as.DNAbin(alignment_dna)
alignment_prot_bin <- as.AAbin(alignment_prot)

#-----------------------------------------------------------
# 4. Convertir a formato phyDat (para phangorn)
#-----------------------------------------------------------
cytb_dna_phy <- phyDat(alignment_dna_bin, type = "DNA")
cytb_prot_phy <- phyDat(alignment_prot_bin, type = "AA")

#-----------------------------------------------------------
# 5. Calcular matrices de distancia evolutiva
#-----------------------------------------------------------

# Modelo Jukes-Cantor (ADN)
dist_dna <- dist.ml(cytb_dna_phy, model = "JC69")

# Modelo JTT (Proteína)
dist_prot <- dist.ml(cytb_prot_phy, model = "JTT")

#-----------------------------------------------------------
# 6. Construir árboles filogenéticos
#-----------------------------------------------------------

# Neighbor-Joining (NJ)
tree_nj_dna <- NJ(dist_dna)
tree_nj_prot <- NJ(dist_prot)

# UPGMA
tree_upgma_dna <- upgma(dist_dna)
tree_upgma_prot <- upgma(dist_prot)

#-----------------------------------------------------------
# 8. Visualización
#-----------------------------------------------------------
par(mfrow = c(2, 2))
plot(tree_nj_dna, main = "CytB ADN - Neighbor Joining (JC69)", cex = 0.7)
plot(tree_upgma_dna, main = "CytB ADN - Upgma", cex = 0.7)
plot(tree_nj_prot, main = "CytB Proteína - Neighbor Joining (JTT)", cex = 0.7)
plot(tree_upgma_prot, main = "CytB Proteína - Upgma", cex = 0.7)



<div class="alert alert-block alert-info">
<b>Ejercicio</b> 
1. Entre a NCBI y Busque el gen "ATP5F1B" de la subunidad beta de ATP sintetasa (Homo sapiens) con identificador: NP_001674.1 y descárguela en formato fasta 

2. Utilice esa secuencia como query para realizar un BLAST. 

3. Escoja entre 5-7 secuencias homólogas de especies diferentes y realice un alineamiento usando https://www.ebi.ac.uk/jdispatcher/msa/clustalo o desde R

4. Usando el alineamiento como input, genere un árbol filogenético con dichas secuencias.

5.  a. Crees que este es un gen conservado? Por qué?
    
    b. Crees que un árbol utilizando la secuencia de nucleótidos cambiaría la topología (la forma y orden) del árbol?
    c. Cómo influye el tipo de modelo (ej WAG, K80, JC69) o la forma de agrupar (Neighbour-joining o UPGMA) en el árbol?

</div>